# Import Libraries

In [1]:
# !pip uninstall pomegranate
# !pip install pomegranate

In [2]:
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator, BayesianEstimator, ParameterEstimator
from pgmpy.inference import VariableElimination
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference.CausalInference import CausalInference
from pgmpy.estimators import HillClimbSearch
from pgmpy.estimators import BicScore
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import daft
from daft import PGM

import sys
import psutil
import os
import gc
import pandas as pd
import numpy as np
import time

# Data Preparation and Cleaning

In [3]:
# Load the Excel file into a DataFrame
file_path = 'Long_COVID_Dataset_AIIMS_BBSR.xlsx'
lc1 = pd.read_excel(file_path, sheet_name='Follow-up at 4 weeks')  # Replace 'Sheet1' with the actual sheet name
lc2=pd.read_excel(file_path, sheet_name='Follow-up at 6 months')

In [4]:
lc1['participant_id'] = lc1['participant_id'].str.upper()
lc2['participant_id'] = lc2['participant_id'].str.upper()

In [5]:
lc1.shape

(487, 50)

In [6]:
# Select the specific columns from lc2
selected_columns = lc2[["participant_id", "longcovid_six", "vaccine_add_dose_six", "vaccine_add_dose_type_six"]]

In [7]:
merged_lc = pd.merge(lc1, selected_columns, on='participant_id', how='outer')

In [8]:
merged_lc.head(5)

,participant_id,Unnamed: 1,date_diff_four,age,sex,education,occupation,occupation_covid,height,weight,...,lc_anxiety_four,lc_depression_four,lc_fever_four,lc_other_four,lc_activitylimit_four,lc_consultation_four,lc_hospitalized_four,longcovid_six,vaccine_add_dose_six,vaccine_add_dose_type_six
0,LC001,NaN,42,29,Male,Post graduation & above,Professional / Technical / Administrative / Ma...,No,167.0,67.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO (End of Data Collection),No,NaN
1,LC002,NaN,42,35,Male,Post graduation & above,Professional / Technical / Administrative / Ma...,No,165.0,80.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LC003,NaN,42,22,Female,College graduate,Unemployed / Student / Homemaker,No,162.0,53.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO (End of Data Collection),No,NaN
3,LC004,NaN,42,23,Female,College graduate,Professional / Technical / Administrative / Ma...,Yes,153.0,60.0,...,No,No,No,Yes,No activity limitation,No,No,NaN,NaN,NaN
4,LC005,NaN,43,22,Female,10th standard or below,Unemployed / Student / Homemaker,No,142.0,50.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO (End of Data Collection),Yes - Second dose,Covishield


In [9]:
merged_lc.shape

(487, 53)

## Extraction of columns from Acute_symptoms symptoms

In [10]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.expand_frame_repr', False)

In [11]:
# Create a DataFrame with binary encoding for the presence of each value
binary_comments =merged_lc['acute_symptoms'].str.get_dummies(sep=',')

In [12]:
# Deleting spaces before column names
binary_comments.columns = [col.lstrip() for col in binary_comments.columns]

In [13]:
#checking stats
binary_comments.sum().sort_values(ascending=False)

Fever                   281
Cough                   182
NO SYMPTOMS             111
Bodyache                 79
Breathing difficulty     70
Loss of taste            63
Loss of smell            58
Tiredness                58
Sore throat              40
Cough                    39
Running nose             37
Fever                    35
Nasal congestion         19
Diarrhoea                11
Headache                 10
Sore throat              10
Bodyache                 10
Breathing difficulty      8
Tiredness                 7
Loss of smell             6
cold                      5
Nasal congestion          4
Vomiting                  4
Running nose              3
Nasal congestion          3
chest pain                2
Loss of taste             2
vomiting                  2
Cold                      2
Diarrhoea                 2
headache                  2
Vomiting                  1
blood in stools           1
Headache                  1
Sudden collapse           1
headache            

In [14]:
binary_comments = binary_comments.sort_index(axis=1)
binary_comments.head(2)

,Bodyache,Bodyache,Breathing difficulty,Breathing difficulty,Cold,Constipation,Cough,Cough,Diarrhoea,Diarrhoea,Diarrhoea,Fever,Fever,Headache,Headache,Headache reeling,Loss of smell,Loss of smell,Loss of taste,Loss of taste,NO SYMPTOMS,Nasal congestion,Nasal congestion,Nasal congestion,Retro orbital pain,Running nose,Running nose,Sneezing,Sore throat,Sore throat,Sudden collapse,Tiredness,Tiredness,Vomiting,Vomiting,Wheezing,blood in stools,chest pain,cold,headache,headache,headche,headreeling,hoarseness of voice,loss of appetite,pain in eyes,vomiting
0,1,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [15]:
# Initialise master list to check if all columns were checked
all_processed_cols = []

In [16]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if 'head' in col_name.lower()]
merged_lc['Headache'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [17]:
binary_comments.iloc[:, cols].columns

Index(['Headache', 'Headache', 'Headache reeling', 'headache', 'headache',
       'headche', 'headreeling'],
      dtype='object')

In [18]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if 'cough' in col_name.lower()]
merged_lc['Cough'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [19]:
binary_comments.iloc[:, cols].columns

Index(['Cough', 'Cough'], dtype='object')

In [20]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if 'fever' in col_name.lower()]
merged_lc['Fever'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [21]:
binary_comments.iloc[:, cols].columns

Index(['Fever', 'Fever'], dtype='object')

In [22]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if 'body' in col_name.lower()]
merged_lc['Bodyache'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [23]:
binary_comments.iloc[:, cols].columns

Index(['Bodyache', 'Bodyache'], dtype='object')

In [24]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if any(sub in col_name.lower() for sub in ['breath', 'wheezing'])]
merged_lc['BreathingDifficulty'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [25]:
binary_comments.iloc[:, cols].columns

Index(['Breathing difficulty', 'Breathing difficulty', 'Wheezing '], dtype='object')

In [26]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if 'taste' in col_name.lower()]
merged_lc['Ageusia'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [27]:
binary_comments.iloc[:, cols].columns

Index(['Loss of taste', 'Loss of taste'], dtype='object')

In [28]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if 'smell' in col_name.lower()]
merged_lc['Anosmia'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [29]:
binary_comments.iloc[:, cols].columns

Index(['Loss of smell', 'Loss of smell'], dtype='object')

In [30]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if 'throat' in col_name.lower()]
merged_lc['SoreThroat'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [31]:
binary_comments.iloc[:, cols].columns

Index(['Sore throat', 'Sore throat'], dtype='object')

In [32]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if 'chest pain' in col_name.lower()]
merged_lc['ChestPain'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [33]:
binary_comments.iloc[:, cols].columns

Index(['chest pain'], dtype='object')

In [34]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if any(sub in col_name.lower() for sub in ['nasal', 'running nose', 'sneezing'])]
merged_lc['NasalIssues'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [35]:
binary_comments.iloc[:, cols].columns

Index(['Nasal congestion', 'Nasal congestion ', 'Nasal congestion ',
       'Running nose', 'Running nose', 'Sneezing'],
      dtype='object')

In [36]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if any(sub in col_name.lower() for sub in ['diarrhoea', 'vomit', 'appetite'])]
merged_lc['DigestiveIssues'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [37]:
binary_comments.iloc[:, cols].columns

Index(['Diarrhoea', 'Diarrhoea ', 'Diarrhoea ', 'Vomiting', 'Vomiting',
       'loss of appetite', 'vomiting'],
      dtype='object')

In [38]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if 'cold' in col_name.lower()]
merged_lc['Cold'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [39]:
binary_comments.iloc[:, cols].columns

Index(['Cold', 'cold'], dtype='object')

In [40]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if 'tired' in col_name.lower()]
merged_lc['Fatigue'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [41]:
binary_comments.iloc[:, cols].columns

Index(['Tiredness', 'Tiredness'], dtype='object')

In [42]:
cols = [i for i, col_name in enumerate(binary_comments.columns) if any(sub in col_name.lower() for sub in ['stools', 'constipation', 'collapse', 'orbital','pain in eyes','voice','collapse'])]
merged_lc['OtherSymptoms'] = binary_comments.iloc[:, cols].any(axis=1).astype(int)
all_processed_cols.extend(cols)

In [43]:
binary_comments.iloc[:, cols].columns

Index(['Constipation', 'Retro orbital pain', 'Sudden collapse',
       'blood in stools', 'hoarseness of voice', 'pain in eyes'],
      dtype='object')

In [44]:
all_processed_names = binary_comments.columns[all_processed_cols].tolist()

not_processed_columns = set(binary_comments.columns) - set(all_processed_names)

if len(not_processed_columns) == 0:
    print("All columns have been processed.")
else:
    print(f"The following columns have not been processed: {', '.join(not_processed_columns)}")

The following columns have not been processed: NO SYMPTOMS 


In [45]:
merged_lc.shape
# From 52 to 67 columns, 15 new columns added

(487, 67)

In [46]:
merged_lc.drop(["participant_id", 'acute_symptoms_count'], axis=1, inplace=True)

## Masking of Variables of object dtype

In [47]:
longcovid = merged_lc.copy()

In [48]:
longcovid.drop(["ct_egene_ngene","ct_orf1a_b_n2gene", "acute_symptoms"], axis=1, inplace=True)

In [49]:
# Drop the first column by column name
longcovid = longcovid.drop(longcovid.columns[0], axis=1)

In [50]:
longcovid.head(5)

,date_diff_four,age,sex,education,occupation,occupation_covid,height,weight,substance_use,smoking,tobacco,alcohol,drugs,past_history_covid,diabetes,hypertension,anxiety,depression,asthma,tb,cancer,other_medical,vaccine_dose_four,vaccine_type_four,vaccine_aefi_four,covid_severity,covid_care,longcovid_four,lc_fatigue_four,lc_cough_four,lc_headache_four,lc_breathing_four,lc_taste_four,lc_smell_four,lc_brainfog_four,lc_chestpain_four,lc_palpitation_four,lc_anxiety_four,lc_depression_four,lc_fever_four,lc_other_four,lc_activitylimit_four,lc_consultation_four,lc_hospitalized_four,longcovid_six,vaccine_add_dose_six,vaccine_add_dose_type_six,Headache,Cough,Fever,Bodyache,BreathingDifficulty,Ageusia,Anosmia,SoreThroat,ChestPain,NasalIssues,DigestiveIssues,Cold,Fatigue,OtherSymptoms
0,42,29,Male,Post graduation & above,Professional / Technical / Administrative / Ma...,No,167.0,67.0,No,NaN,NaN,NaN,NaN,Yes,No,No,No,No,No,No,No,NaN,Yes - Two doses,Covaxin,Yes - Mild,Mild/Moderate - Did not receive oxygen,Home isolation,NO (End of Data Collection),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO (End of Data Collection),No,NaN,0,1,1,1,1,0,1,0,0,0,0,0,1,0
1,42,35,Male,Post graduation & above,Professional / Technical / Administrative / Ma...,No,165.0,80.0,No,NaN,NaN,NaN,NaN,No,No,No,No,No,No,No,No,NaN,Yes - Two doses,Covaxin,No,Mild/Moderate - Did not receive oxygen,Home isolation,NO (End of Data Collection),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,1,1,0,0,0,0,0,0,0
2,42,22,Female,College graduate,Unemployed / Student / Homemaker,No,162.0,53.0,No,NaN,NaN,NaN,NaN,No,No,No,No,No,No,No,No,NaN,Yes - Two doses,Covaxin,No,Mild/Moderate - Did not receive oxygen,Home isolation,NO (End of Data Collection),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO (End of Data Collection),No,NaN,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,42,23,Female,College graduate,Professional / Technical / Administrative / Ma...,Yes,153.0,60.0,No,NaN,NaN,NaN,NaN,No,No,No,No,No,No,No,No,NaN,Yes - Two doses,Covaxin,No,Mild/Moderate - Did not receive oxygen,Home isolation,YES - Not severe,Yes,Yes,No,No,No,No,No,No,No,No,No,No,Yes,No activity limitation,No,No,NaN,NaN,NaN,0,0,1,1,0,0,1,1,0,0,0,0,1,0
4,43,22,Female,10th standard or below,Unemployed / Student / Homemaker,No,142.0,50.0,No,NaN,NaN,NaN,NaN,No,No,No,No,No,No,No,No,NaN,Yes - Only first dose,Covaxin,No,Mild/Moderate - Did not receive oxygen,Home isolation,NO (End of Data Collection),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO (End of Data Collection),Yes - Second dose,Covishield,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [51]:
longcovid.dtypes

date_diff_four                 int64
age                            int64
sex                           object
education                     object
occupation                    object
occupation_covid              object
height                       float64
weight                       float64
substance_use                 object
smoking                       object
tobacco                       object
alcohol                       object
drugs                         object
past_history_covid            object
diabetes                      object
hypertension                  object
anxiety                       object
depression                    object
asthma                        object
tb                            object
cancer                        object
other_medical                 object
vaccine_dose_four             object
vaccine_type_four             object
vaccine_aefi_four             object
covid_severity                object
covid_care                    object
l

In [52]:
# Filter columns with data type float
float_columns = longcovid.select_dtypes(include=['float','object'])

# Loop through each float column and print unique values
for col in float_columns.columns:
    print(f"{col}: {longcovid[col].unique()}\n")

sex: ['Male' 'Female']

education: ['Post graduation & above ' 'College graduate' '10th standard or below'
 'Higher Secondary (11 to 12th standard)'
 'Illiterate / No formal schooling ']

occupation: ['Professional / Technical / Administrative / Managerial '
 'Unemployed / Student / Homemaker ' 'Skilled and Unskilled manual '
 'Other' 'Retired ']

occupation_covid: ['No' 'Yes']

height: [167. 165. 162. 153. 142. 163. 148. 168. 152. 155. 154. 151. 178. 150.
 166. 180. 175. 146. 170. 160. 156. 176. 159. 145. 147. 158. 169. 184.
 172. 161. 149. 179. 157. 164. 171. 182.  nan 174. 177. 173. 140.]

weight: [ 67.  80.  53.  60.  50.  62.  45.  56.  72.  76.  54.  52.  74.  66.
  78.  84.  73.  85.  49.  68.  65.  61.  71.  64.  69.  58.  51.  82.
  70.  90.  41.  59.  75.  55.  57.  63.  86.  40.  87.  48.  95.  79.
 103.  42.  47.  43.  nan  30.  44.  96.  25.  77.  81.]

substance_use: ['No' 'Yes']

smoking: [nan 'Current' 'Former (Not smoked more than 1 year)' 'Never'
 'No response ']

tob

In [53]:
# Define mapping dictionaries for each categorical variable
gender_mapping = {'Female': 0, 'Male': 1}
longcovid['sex'] = longcovid['sex'].map(gender_mapping)

In [54]:
education_mapping = {
    'Post graduation & above ': 0,
    'College graduate': 1,
    '10th standard or below': 2,
    'Higher Secondary (11 to 12th standard)': 3,
    'Illiterate / No formal schooling ': 4
}

longcovid['education'] = longcovid['education'].map(education_mapping)

occupation_mapping = {'Professional / Technical / Administrative / Managerial ': 0, 'Unemployed / Student / Homemaker ': 1, 'Skilled and Unskilled manual ':2, 'Other':3, 'Retired ':4}
longcovid['occupation'] = longcovid['occupation'].map(occupation_mapping)

occupation_covid_mapping = {'No': 0, 'Yes': 1}
longcovid['occupation_covid'] = longcovid['occupation_covid'].map(occupation_covid_mapping)

# Substance use mappings
smoking_mapping = {np.nan: 0, 'Current':1, 'Former (Not smoked more than 1 year)':2, 'Never':0, 'No response ':3}
longcovid['smoking'] = longcovid['smoking'].map(smoking_mapping) 

tobacco_mapping= {np.nan:0, 'No':0, 'Yes':1}
longcovid['tobacco'] = longcovid['tobacco'].map(tobacco_mapping) 

longcovid['smoking'] = np.where((longcovid['smoking'] == 1) | (longcovid['tobacco'] == 1), 1, 0)
longcovid.drop('tobacco', axis=1, inplace=True)

alcohol_mapping = {np.nan:0, 'No':0, 'Yes ':1}
longcovid['alcohol'] = longcovid['alcohol'].map(alcohol_mapping) 

drugs_mapping = {np.nan:0, 'No':0, 'Yes':1}
longcovid['drugs'] = longcovid['drugs'].map(drugs_mapping) 

longcovid.drop(['substance_use'], axis=1, inplace=True)

# Historical and medical conditions mappings
past_history_covid_mapping = {'Yes': 1, 'No': 0}
longcovid['past_history_covid'] = longcovid['past_history_covid'].map(past_history_covid_mapping)

# Map for all 'Yes' 'No' columns 
yes_no_mapping = {np.nan: 0, 'No': 0, 'Yes': 1}

for column in ['diabetes', 'hypertension', 'anxiety', 'depression', 'asthma', 'cancer', 'other_medical']:
    longcovid[column] = longcovid[column].map(yes_no_mapping)
    
tb_mapping = {'No': 0, 'Yes - Previous TB': 1, 'Yes - Active TB': 2}
longcovid['tb'] = longcovid['tb'].map(tb_mapping)

covid_severity_mapping = {'Mild/Moderate - Did not receive oxygen': 0,
 'Severe - Received oxygen or was told you require oxygen':1,
 'Critical - Received invasive ventilation':2}
longcovid['covid_severity'] = longcovid['covid_severity'].map(covid_severity_mapping)

covid_care_mapping = {'Home isolation': 0, 'Admitted to hospital': 1}
longcovid['covid_care'] = longcovid['covid_care'].map(covid_care_mapping)

# Symptoms
symptoms_mapping = {np.nan: 0, 'Yes': 1, 'No':0}
for column in ['lc_fatigue_four', 'lc_cough_four', 'lc_headache_four', 'lc_breathing_four', 'lc_taste_four', 'lc_smell_four', 'lc_brainfog_four', 'lc_chestpain_four', 'lc_palpitation_four', 'lc_anxiety_four', 'lc_depression_four', 'lc_fever_four', 'lc_other_four']:
    longcovid[column] = longcovid[column].map(symptoms_mapping)

lc_activitylimit_four_mapping = {np.nan:0, 'No activity limitation': 0, 'Activities limited a little':1, 'Activities limited a lot':2}
longcovid['lc_activitylimit_four'] = longcovid['lc_activitylimit_four'].map(lc_activitylimit_four_mapping)

longcovid_six_mapping = {'NO (End of Data Collection)':0,  np.nan:0, 'YES - A little':1, 'YES - A lot':2}
longcovid['longcovid_six'] = longcovid['longcovid_six'].map(longcovid_six_mapping)

longcovid_four_mapping = {np.nan:0, 'NO (End of Data Collection)': 0, 'YES - Not severe': 1, 'YES - Severe': 2}
longcovid['longcovid_four'] = longcovid['longcovid_four'].map(longcovid_four_mapping)

# lc_consultation_four mapping
lc_consultation_four_mapping = {np.nan: 0, 'No': 0, 'Yes': 1}
longcovid['lc_consultation_four'] = longcovid['lc_consultation_four'].map(lc_consultation_four_mapping)

# lc_hospitalized_four mapping
lc_hospitalized_four_mapping = {np.nan: 0, 'No': 0, 'Yes': 1}
longcovid['lc_hospitalized_four'] = longcovid['lc_hospitalized_four'].map(lc_hospitalized_four_mapping)


In [55]:
# date_diff_7_groups = {
#     range(28, 35): 0,
#     range(35, 42): 1,
#     range(42, 49): 2,
#     range(49, 56): 3,
#     range(56, 63): 4,
#     range(63, 70): 5,
#     range(70, 77): 6
# }

# longcovid['date_diff_four_7_classes'] = longcovid['date_diff_four'].map(lambda x: next((v for k, v in date_diff_7_groups.items() if x in k), None))

date_diff_4_groups = {
    range(28, 40): 0,
    range(40, 50): 1,
    range(50, 60): 2,
    range(60, 74): 3
}

longcovid['date_diff_four_4_classes'] = longcovid['date_diff_four'].map(lambda x: next((v for k, v in date_diff_4_groups.items() if x in k), None))
longcovid.drop(['date_diff_four'], axis=1, inplace=True)

In [56]:
# Mapping for doses
def map_dose(row):
    dose_mapping = {'No': 0, 'Yes - Only first dose': 1, 'Yes - Two doses': 2, 'Yes - Precaution dose': 3, 'Yes - Second dose':2, 'Yes - First dose':1}
    
    # If vaccine_add_dose_six is 'No' or NaN, then consider the vaccine_dose_four value.
    dose_six = dose_mapping.get(row['vaccine_add_dose_six'], 0)
    dose_four = dose_mapping.get(row['vaccine_dose_four'], 0)
    
    # Return the maximum dose from the two columns
    return max(dose_six, dose_four)

longcovid['vaccine_dose'] = longcovid.apply(map_dose, axis=1)

# Mapping for vaccine types
def map_type(row):
    if row['vaccine_dose'] == 0:  # If no dose, return 0
        return 0
    # If the two types are the same Covishield:1 Covaxin:2
    if row['vaccine_type_four'] == row['vaccine_add_dose_type_six']:
        return {'Covishield': 1, 'Covaxin': 2}.get(row['vaccine_type_four'], 0)
    # If the types are different or one is NaN
    elif (row['vaccine_type_four'] == 'Covishield' and row['vaccine_add_dose_type_six'] == 'Covaxin') or \
         (row['vaccine_type_four'] == 'Covaxin' and row['vaccine_add_dose_type_six'] == 'Covishield'):
        return 3
    else:  # Return the value that's not NaN
        return {'Covishield': 1, 'Covaxin': 2}.get(row['vaccine_type_four'] or row['vaccine_add_dose_type_six'], 0)

longcovid['vaccine_type'] = longcovid.apply(map_type, axis=1)

# Map vaccine_aefi_four
effect_mapping = {'Yes - Mild ': 1, 'No': 0, 'Yes - Moderate - Consulted a doctor': 2, 'Yes - Severe and Hospital admission ': 3}
longcovid['vaccine_effect'] = longcovid['vaccine_aefi_four'].map(effect_mapping).fillna(0).astype(int)

# Dropping encoded columns
longcovid.drop(['vaccine_aefi_four','vaccine_type_four','vaccine_add_dose_six','vaccine_add_dose_type_six', 'vaccine_dose_four' ], axis=1, inplace=True)


In [57]:
#Age ranges
age_groups = {
    range(0, 13): 0, #'0-12',
    range(13, 25): 1, #'13-24',
    range(25, 45): 2, #'25-44',
    range(45, 66): 3, #'45-65',
    range(66, 120): 4, #'66+'
}

longcovid['Age_group'] =longcovid['age'].map(lambda Age: next((a for b, a in age_groups.items() if Age in b), None))
longcovid.drop(['age'], axis=1, inplace=True)

In [58]:
# Calculate BMI, but set it to NaN if either height or weight is NaN
longcovid['BMI'] = np.where(longcovid['height'].notna() & longcovid['weight'].notna(), 
                            longcovid['weight'] / (longcovid['height']/100)**2, 
                            np.nan)

# Determine obesity(body type) based on BMI category
longcovid['obesity'] = np.where(longcovid['BMI'] >= 30, 1, 
                                np.where(longcovid['BMI'] < 30, 0, np.nan))

In [59]:
# Calculate mean BMI for each combination of age and sex
mean_bmi = longcovid.groupby(['Age_group', 'sex'])['BMI'].transform('mean')

# Replace NaN values in BMI with the corresponding mean value
longcovid['BMI'].fillna(mean_bmi, inplace=True)

# Recalculate the 'obesity' column based on the updated 'BMI' values
longcovid['obesity'] = np.where(longcovid['BMI'] >= 30, 1, 0)

longcovid.drop(['BMI', 'height', 'weight'], axis=1, inplace=True)

In [60]:
longcovid.isna().sum()[longcovid.isna().sum() != 0]

Series([], dtype: int64)

In [61]:
# Checking for variability
longcovid.sum()[longcovid.var() == 0]

depression              0
lc_depression_four      0
lc_hospitalized_four    0
dtype: int64

In [62]:
#10 columns
rmcol = ['DigestiveIssues', 'OtherSymptoms','Bodyache','Fatigue','NasalIssues','anxiety', 'lc_headache_four', 'lc_brainfog_four','lc_palpitation_four','lc_anxiety_four']

In [63]:
# longcovid.var() <0.01

In [64]:
# Get the columns with no variability
cols_to_drop = longcovid.columns[longcovid.var() == 0].tolist()
# Drop columns from the dataframe
longcovid.drop(columns=cols_to_drop, inplace=True)

In [65]:
longcovid.to_csv("longcovid.csv", index=False)

In [66]:
longcovid.describe()

,sex,education,occupation,occupation_covid,smoking,alcohol,drugs,past_history_covid,diabetes,hypertension,anxiety,asthma,tb,cancer,other_medical,covid_severity,covid_care,longcovid_four,lc_fatigue_four,lc_cough_four,lc_headache_four,lc_breathing_four,lc_taste_four,lc_smell_four,lc_brainfog_four,lc_chestpain_four,lc_palpitation_four,lc_anxiety_four,lc_fever_four,lc_other_four,lc_activitylimit_four,lc_consultation_four,longcovid_six,Headache,Cough,Fever,Bodyache,BreathingDifficulty,Ageusia,Anosmia,SoreThroat,ChestPain,NasalIssues,DigestiveIssues,Cold,Fatigue,OtherSymptoms,date_diff_four_4_classes,vaccine_dose,vaccine_type,vaccine_effect,Age_group,obesity
count,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000,487.000000
mean,0.591376,1.583162,1.151951,0.061602,0.084189,0.039014,0.002053,0.036961,0.119097,0.104723,0.006160,0.030801,0.012320,0.039014,0.108830,0.156057,0.225873,0.314168,0.188912,0.094456,0.004107,0.049281,0.012320,0.008214,0.006160,0.024641,0.006160,0.006160,0.002053,0.051335,0.125257,0.100616,0.075975,0.034908,0.453799,0.648871,0.182752,0.162218,0.133470,0.131417,0.102669,0.004107,0.114990,0.043121,0.014374,0.133470,0.010267,0.845996,1.747433,1.369610,0.086242,2.217659,0.047228
std,0.492085,1.034768,1.076073,0.240678,0.277956,0.193828,0.045314,0.188860,0.324235,0.306511,0.078325,0.172955,0.127706,0.193828,0.311745,0.385273,0.418586,0.511055,0.391841,0.292763,0.064018,0.216677,0.110425,0.090348,0.078325,0.155187,0.078325,0.078325,0.045314,0.220906,0.388517,0.301129,0.280318,0.183735,0.498373,0.477814,0.386860,0.369029,0.340432,0.338203,0.303839,0.064018,0.319338,0.203339,0.119148,0.340432,0.100908,0.659607,0.790809,0.857634,0.315504,0.815145,0.212344
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,1.000000,0.000000,2.000000,0.000000
50%,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2.000000,2.000000,0.000000,2.000000,0.000000
75%,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.00000

In [67]:
longcovid.shape

(487, 53)

In [68]:
longcovid.head(2)

,sex,education,occupation,occupation_covid,smoking,alcohol,drugs,past_history_covid,diabetes,hypertension,anxiety,asthma,tb,cancer,other_medical,covid_severity,covid_care,longcovid_four,lc_fatigue_four,lc_cough_four,lc_headache_four,lc_breathing_four,lc_taste_four,lc_smell_four,lc_brainfog_four,lc_chestpain_four,lc_palpitation_four,lc_anxiety_four,lc_fever_four,lc_other_four,lc_activitylimit_four,lc_consultation_four,longcovid_six,Headache,Cough,Fever,Bodyache,BreathingDifficulty,Ageusia,Anosmia,SoreThroat,ChestPain,NasalIssues,DigestiveIssues,Cold,Fatigue,OtherSymptoms,date_diff_four_4_classes,vaccine_dose,vaccine_type,vaccine_effect,Age_group,obesity
0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,1,0,0,0,0,0,1,0,1,2,2,1,2,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,2,2,0,2,0


In [69]:
longcovid.columns

Index(['sex', 'education', 'occupation', 'occupation_covid', 'smoking',
       'alcohol', 'drugs', 'past_history_covid', 'diabetes', 'hypertension',
       'anxiety', 'asthma', 'tb', 'cancer', 'other_medical', 'covid_severity',
       'covid_care', 'longcovid_four', 'lc_fatigue_four', 'lc_cough_four',
       'lc_headache_four', 'lc_breathing_four', 'lc_taste_four',
       'lc_smell_four', 'lc_brainfog_four', 'lc_chestpain_four',
       'lc_palpitation_four', 'lc_anxiety_four', 'lc_fever_four',
       'lc_other_four', 'lc_activitylimit_four', 'lc_consultation_four',
       'longcovid_six', 'Headache', 'Cough', 'Fever', 'Bodyache',
       'BreathingDifficulty', 'Ageusia', 'Anosmia', 'SoreThroat', 'ChestPain',
       'NasalIssues', 'DigestiveIssues', 'Cold', 'Fatigue', 'OtherSymptoms',
       'date_diff_four_4_classes', 'vaccine_dose', 'vaccine_type',
       'vaccine_effect', 'Age_group', 'obesity'],
      dtype='object')

# Bayesian Model

In [ ]:
longcovid1 = pd.read_csv('longcovid.csv')